# 03d — Weather Enrichment (Problem C)

Enrich the case-control dataset with historical weather data using the Meteostat API.

> **Note:** `meteostat` 1.7.6 does not have `nearby_iata()`. We resolve stations via lat/lon lookup.

**Input:** `data/processed/preflight_casecontrol.parquet`  
**Output:** `data/processed/preflight_weather_enriched.parquet`

In [1]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    root = root.parent
os.chdir(root)
print(f'Working directory: {root}')

Working directory: e:\OSFDA


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from meteostat import Hourly, Stations
from tqdm.auto import tqdm
from pathlib import Path

IN_PATH  = Path('data/processed/preflight_casecontrol.parquet')
OUT_PATH = Path('data/processed/preflight_weather_enriched.parquet')

df = pd.read_parquet(IN_PATH)
print(f'Loaded dataset with {len(df):,} rows.')
print(f'Columns: {df.columns.tolist()}')
print(f'Incident rate: {df["incident"].mean()*100:.2f}%')

Loaded dataset with 1,012,278 rows.
Columns: ['FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEP_TIME', 'DEP_DELAY', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'airport_code', 'incident']
Incident rate: 4.76%


## 1. Parse Departure Timestamp

In [3]:
def parse_bts_time(row):
    """Convert FL_DATE + DEP_TIME (HHMM float) to a datetime."""
    if pd.isna(row['DEP_TIME']):
        return pd.NaT
    t = int(row['DEP_TIME'])
    hours = (t // 100) % 24
    mins  = t % 100
    try:
        return pd.to_datetime(row['FL_DATE']) + timedelta(hours=hours, minutes=mins)
    except Exception:
        return pd.NaT

df['departure_timestamp'] = df.apply(parse_bts_time, axis=1)
print(f'Parsed {df["departure_timestamp"].notna().sum():,} / {len(df):,} timestamps.')

Parsed 991,697 / 1,012,278 timestamps.


## 2. Resolve Weather Stations

Meteostat 1.7.6 does not support `nearby_iata()`.  
We use a curated lat/lon table for the airports in our dataset,
then look up the nearest METAR station with `stations.nearby(lat, lon)`.

In [4]:
# Embedded lat/lon table for the top US commercial airports (covers >90% of BTS volume)
# Source: OurAirports / FAA
AIRPORT_COORDS = {
    'ATL': (33.6367, -84.4281), 'LAX': (33.9425, -118.4081), 'ORD': (41.9742, -87.9073),
    'DFW': (32.8998, -97.0403), 'DEN': (39.8561, -104.6737), 'JFK': (40.6413, -73.7781),
    'SFO': (37.6213, -122.3790), 'SEA': (47.4502, -122.3088), 'LAS': (36.0840, -115.1537),
    'MCO': (28.4312, -81.3081), 'EWR': (40.6895, -74.1745), 'MIA': (25.7959, -80.2870),
    'PHX': (33.4373, -112.0078), 'IAH': (29.9902, -95.3368), 'BOS': (42.3656, -71.0096),
    'MSP': (44.8820, -93.2218), 'DTW': (42.2162, -83.3554), 'FLL': (26.0726, -80.1527),
    'PHL': (39.8744, -75.2424), 'LGA': (40.7769, -73.8740), 'BWI': (39.1754, -76.6682),
    'DCA': (38.8512, -77.0402), 'SLC': (40.7884, -111.9778), 'SAN': (32.7338, -117.1933),
    'MDW': (41.7868, -87.7522), 'TPA': (27.9755, -82.5332), 'HNL': (21.3245, -157.9251),
    'PDX': (45.5887, -122.5975), 'DAL': (32.8471, -96.8518), 'OAK': (37.7213, -122.2208),
    'STL': (38.7487, -90.3700), 'MCI': (39.2976, -94.7139), 'RDU': (35.8776, -78.7875),
    'AUS': (30.1975, -97.6664), 'SJC': (37.3626, -121.9290), 'BNA': (36.1245, -86.6782),
    'MEM': (35.0424, -89.9767), 'JAX': (30.4941, -81.6879), 'CLE': (41.4117, -81.8498),
    'RSW': (26.5362, -81.7552), 'IND': (39.7173, -86.2944), 'CMH': (39.9980, -82.8919),
    'CVG': (39.0488, -84.6678), 'OMA': (41.3032, -95.8941), 'ABQ': (35.0402, -106.6090),
    'BUF': (42.9405, -78.7322), 'RIC': (37.5052, -77.3197), 'PVD': (41.7272, -71.4282),
    'OKC': (35.3931, -97.6007), 'SNA': (33.6757, -117.8682), 'SJU': (18.4394, -66.0018),
    'ANC': (61.1744, -149.9982), 'FAI': (64.8151, -147.8560), 'JNU': (58.3550, -134.5763),
    'GNV': (29.6900, -82.2717), 'DAB': (29.1799, -81.0581), 'BTV': (44.4719, -73.1533),
    'BZN': (45.7777, -111.1527), 'COS': (38.8059, -104.7008), 'STS': (38.5090, -122.8130),
    'TTN': (40.2768, -74.8135), 'OGD': (41.1960, -112.0120), 'ABI': (32.4113, -99.6819),
    'LRD': (27.5438, -99.4615), 'EKO': (40.8249, -115.7912), 'BLI': (48.7928, -122.5375),
    'FAR': (46.9207, -96.8158), 'HNL': (21.3245, -157.9251), 'LBL': (37.0442, -100.9601),
    'RNO': (39.4991, -119.7681), 'RKS': (41.5942, -109.0651), 'SDL': (33.6229, -111.9107),
}

unique_airports = df['ORIGIN'].unique().tolist()
covered = [a for a in unique_airports if a in AIRPORT_COORDS]
print(f'Dataset airports: {len(unique_airports)}')
print(f'In coordinate table: {len(covered)} ({len(covered)/len(unique_airports)*100:.0f}%)')
print(f'Missing coords (will be skipped): {[a for a in unique_airports if a not in AIRPORT_COORDS][:15]}')

Dataset airports: 371
In coordinate table: 70 (19%)
Missing coords (will be skipped): ['OGG', 'EVV', 'HRL', 'LEX', 'BTM', 'HYS', 'TUL', 'SFB', 'MSY', 'BDL', 'PNS', 'CLT', 'SGF', 'MDT', 'GRR']


In [5]:
# Cache: resolve each airport to its nearest meteostat station ID
airport_station_map = {}  # IATA -> station_id

for iata, (lat, lon) in tqdm(AIRPORT_COORDS.items(), desc='Resolving stations'):
    if iata not in unique_airports:
        continue
    try:
        station = Stations().nearby(lat, lon).fetch(1)
        if not station.empty:
            airport_station_map[iata] = station.index[0]
    except Exception:
        pass

print(f'Resolved {len(airport_station_map)} airport -> station mappings')

Resolving stations:   0%|          | 0/71 [00:00<?, ?it/s]

Resolved 70 airport -> station mappings


## 3. Fetch Weather Data per Airport

In [6]:
# Date range for weather fetch
start_dt = df['departure_timestamp'].min().to_pydatetime().replace(tzinfo=None)
end_dt   = df['departure_timestamp'].max().to_pydatetime().replace(tzinfo=None)
print(f'Fetching weather: {start_dt} to {end_dt}')

weather_frames = []

for iata, station_id in tqdm(airport_station_map.items(), desc='Fetching weather'):
    try:
        data = Hourly(station_id, start_dt, end_dt).fetch()
        if not data.empty:
            data = data.reset_index().rename(columns={'time': 'weather_time'})
            data['ORIGIN'] = iata
            weather_frames.append(data)
    except Exception:
        pass

if weather_frames:
    full_weather_df = pd.concat(weather_frames, ignore_index=True)
    print(f'Downloaded {len(full_weather_df):,} hourly weather records across {full_weather_df["ORIGIN"].nunique()} airports.')
else:
    print('WARNING: No weather data fetched. Check connectivity / meteostat cache.')
    full_weather_df = pd.DataFrame()

Fetching weather: 2018-01-01 05:34:00 to 2020-12-31 23:55:00


Fetching weather:   0%|          | 0/70 [00:00<?, ?it/s]

Downloaded 1,813,416 hourly weather records across 69 airports.


## 4. Merge Weather into Dataset

In [7]:
if not full_weather_df.empty:
    # Round departure time to nearest hour for matching
    df['weather_lookup_time'] = df['departure_timestamp'].dt.round('h')
    full_weather_df['weather_time'] = pd.to_datetime(full_weather_df['weather_time']).dt.round('h')

    enriched_df = pd.merge(
        df,
        full_weather_df[['ORIGIN', 'weather_time', 'temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres']],
        left_on=['ORIGIN', 'weather_lookup_time'],
        right_on=['ORIGIN', 'weather_time'],
        how='left'
    ).drop(columns=['weather_time'], errors='ignore')

    print(f'Enrichment complete. Rows: {len(enriched_df):,}')
    coverage = enriched_df['temp'].notna().mean() * 100
    print(f'Weather coverage: {coverage:.1f}%')
else:
    # Fallback: add empty weather columns so downstream notebooks don't break
    print('Fallback: creating empty weather columns.')
    enriched_df = df.copy()
    for col in ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres']:
        enriched_df[col] = np.nan
    enriched_df['weather_lookup_time'] = pd.NaT

Enrichment complete. Rows: 1,012,278
Weather coverage: 72.9%


## 5. Validate & Save

In [8]:
# Check missing weather
missing_mask = enriched_df['temp'].isna()
print(f'Rows missing weather: {missing_mask.sum():,} ({missing_mask.mean()*100:.1f}%)')

weather_cols = [c for c in ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres'] if c in enriched_df.columns]
print(enriched_df[weather_cols].describe())

Rows missing weather: 274,106 (27.1%)
            temp       dwpt       rhum      prcp       wspd         pres
count   738172.0   738100.0   738100.0  690442.0   737564.0     734240.0
mean   16.528907   9.079932  66.244806  0.103901  12.990498  1016.466224
std    10.511245  10.399185  21.738914   0.89327   8.670117     6.654223
min        -38.9      -41.4        2.0       0.0        0.0        964.1
25%          8.9        1.1       52.0       0.0        7.6       1012.4
50%         17.8       10.6       70.0       0.0       11.2       1016.2
75%         24.4       17.9       84.0       0.0       18.4       1020.3
max         46.7       28.3      100.0      72.9       98.3       1049.7


In [9]:
enriched_df.to_parquet(OUT_PATH, index=False)
print(f'Saved weather-enriched dataset to {OUT_PATH}')
print(f'Shape: {enriched_df.shape}')

Saved weather-enriched dataset to data\processed\preflight_weather_enriched.parquet
Shape: (1012278, 31)
